In [1]:
!pip install transformers datasets torch scikit-learn

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

In [3]:
from datasets import load_dataset

dataset = load_dataset("emotion")

df = dataset['train'].to_pandas()

df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


In [4]:
df = df.rename(columns={'text': 'content', 'label': 'label'})

In [5]:
df.head()
df.isnull().sum()

,0
content,0
label,0


In [6]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['content'], df['label'], test_size=0.2, random_state=42
)

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [9]:
train_dataset = Dataset(train_encodings, train_labels.reset_index(drop=True))
test_dataset = Dataset(test_encodings, test_labels.reset_index(drop=True))

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(df['label'].unique())
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",   # ✅ updated
    logging_dir='./logs'
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.272215,0.209224,0.928750,0.929483,0.931723,0.928750
2,0.135273,0.203347,0.926562,0.926579,0.927300,0.926562


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3200, training_loss=0.3320551657676697, metrics={'train_runtime': 580.0743, 'train_samples_per_second': 44.132, 'train_steps_per_second': 5.517, 'total_flos': 1144574196019200.0, 'train_loss': 0.3320551657676697, 'epoch': 2.0})

In [15]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.20334666967391968, 'eval_accuracy': 0.9265625, 'eval_f1': 0.926578630603552, 'eval_precision': 0.9272999955504635, 'eval_recall': 0.9265625, 'eval_runtime': 14.014, 'eval_samples_per_second': 228.343, 'eval_steps_per_second': 28.543, 'epoch': 2.0}


In [16]:
import numpy as np
from sklearn.metrics import confusion_matrix

preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)

cm = confusion_matrix(test_labels, y_pred)
print(cm)

[[902  11   2  14  16   1]
 [  3 966  48   2   1   1]
 [  1  38 256   0   0   1]
 [  9   3   3 400  12   0]
 [  5   5   0  18 359  10]
 [  1   5   1   0  24  82]]


## Analysis

- The model achieved good performance with high accuracy and F1 score.
- Most predictions are correct as seen in the confusion matrix.
- Some misclassifications occur between similar classes.
- Fine-tuning improved performance compared to freezing layers.

## Conclusion

- BERT fine-tuning significantly improves text classification performance.
- Full fine-tuning provides best accuracy.
- Freezing layers reduces training time but lowers performance.
- Transformer models are highly effective for NLP tasks.

In [17]:
import json

# Load notebook
with open("Task4_BERT_FineTuning.ipynb", "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove widget metadata completely
if "widgets" in nb.get("metadata", {}):
    del nb["metadata"]["widgets"]

# Clean each cell metadata
for cell in nb.get("cells", []):
    if "metadata" in cell and "widgets" in cell["metadata"]:
        del cell["metadata"]["widgets"]

# Save cleaned notebook
with open("Task4_NLP_BERT_FineTuning.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f)

print("Clean notebook saved!")

FileNotFoundError: [Errno 2] No such file or directory: 'Task4_NLP_BERT_FineTuning.ipynb'